# 02. Hybrid Retrieval & Cross-Encoder Benchmark

Compares pure Dense Semantic Retrieval (ChromaDB) vs Sparse Lexical Retrieval (BM25) vs Reciprocal Rank Fusion (RRF) with FlashRank re-ranking.

In [ ]:
from pathlib import Path
import json, sys
sys.path.append('..')

from src.retrieval.vector_store import ChromaVectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_reranker import HybridReranker

# Load indexed chunks
jsonl_path = Path('../data/processed/apple_10k_sample.jsonl')
chunks = [json.loads(line) for line in open(jsonl_path, encoding='utf-8') if line.strip()]

vector_store = ChromaVectorStore()
bm25 = BM25Retriever(chunks)
reranker = HybridReranker()

query = 'What was the Net Income from 2021 to 2023?'
print(f'Test Query: {query}')

In [ ]:
# Compare Retrieval Stages
dense_res = vector_store.search(query, top_k=5)
sparse_res = bm25.search(query, top_k=5)
fused = reranker.reciprocal_rank_fusion(dense_res, sparse_res, top_k=5)
reranked = reranker.rerank(query, fused, top_k=3)

print('Dense Top Page:', dense_res[0][0]['page_number'])
print('BM25 Top Page:', sparse_res[0][0]['page_number'])
print('RRF Top Page:', fused[0]['page_number'])
print('Reranked Top Result Page:', reranked[0]['page_number'], 'Score:', reranked[0].get('rerank_score'))